# 🧠 CerebroSpheroid V5: Gemma 2 to Cerebro Distillation
### Google Gemma 2 (2B / 9B) Teacher • CerebroSpheroid V5 Small (105.9M) Student • AALW Distillation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjitKumar-git/cerebro_spheroid_v5/blob/main/colab/gemma2cerebro.ipynb)

- **Teacher Model**: Google Gemma 2 2B (`google/gemma-2-2b-it` / `unsloth/gemma-2-2b-it`). Loaded in 4-bit NormalFloat (BitsAndBytes) requiring only ~2.5 GB VRAM.
- **Student Model**: CerebroSpheroid V5 Small (105.9M parameters, 16 cortical shells, 40 shortcut tracts, 32,000 vocab).
- **Answer-Anchor Loss Weighting (AALW)**: 0x on prompt, 3.5x on deliberative thoughts (`<think> ... </think>`), 4.0x on tool actions (`<act> ... </act>`), and 5.0x on final answer tokens.
- **Option A Cloud Mirroring**: Lightweight weights-only checkpoints (~229 MB) saved to Google Drive and synced non-blockingly to `bioflo.in`.

---
### ⚡ Colab Hardware Setting:
**Runtime** -> **Change runtime type** -> **T4 GPU** (or A100 / L4 GPU)

## ⚡ Step 1: System Dependencies, Hugepages & Repository Sync
Installs dependencies, detects GPU acceleration, configures tokens, and pulls the latest codebase from GitHub.

In [ ]:
# ==============================================================================
# 🚀 Step 1: System Dependencies, Hugepages & Repository Sync
# ==============================================================================
!echo always 2>/dev/null | sudo -n tee /sys/kernel/mm/transparent_hugepage/enabled >/dev/null 2>&1 || true
!pip install -q transformers datasets sentencepiece accelerate bitsandbytes huggingface_hub torch

import os, sys, subprocess, importlib, torch

# 1. Verify GPU Acceleration
if torch.cuda.is_available():
    dev_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"⚡ CUDA GPU Active: {dev_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("⚠️ Running on CPU. In Colab, select: Runtime -> Change runtime type -> T4 GPU.")

# 2. Synchronize Repository to latest origin/main
REPO_DIR = "/content/cerebro_v5"
if not os.path.exists(REPO_DIR):
    print("📥 Cloning cerebro_spheroid_v5 repository...")
    subprocess.run(["git", "clone", "https://github.com/AjitKumar-git/cerebro_spheroid_v5.git", REPO_DIR], check=True)
else:
    print("🔄 Pulling latest updates from origin/main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Clear module caches
importlib.invalidate_caches()
for mod in list(sys.modules.keys()):
    if mod.startswith(("src", "training", "autotune", "colab")):
        del sys.modules[mod]

# 3. Optional Google Drive Mounting (Persistent Checkpoints Across Disconnects)
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
        print("✅ Google Drive mounted: Checkpoints will persist permanently!")
except Exception:
    print("ℹ️ Google Drive not mounted. Using local /content/checkpoints.")

## ⚡ Step 2: Configure & Launch Distillation
Executes teacher-to-student distillation using Gemma 2 (2B / 9B) with Answer-Anchor Loss Weighting and automatic Option A cloud sync.

In [ ]:
# ==============================================================================
# ⚡ Step 2: Launch Gemma 2 -> Cerebro V5 Small Distillation
# ==============================================================================
from colab.distill_gemma_to_v5_small import run_distillation

# Primary Distillation Hyperparameters
TEACHER_MODEL   = "google/gemma-2-2b-it"   # Or "unsloth/gemma-2-2b-it", "google/gemma-2-9b-it"
DISTILL_MODE    = "stream"                 # "stream" (fast frontier dataset), "live" (on-policy), "hybrid"
TOTAL_STEPS     = 5000                     # Total distillation steps
BATCH_SIZE      = 4                        # Micro-batch size per step
GRAD_ACCUM      = 4                        # Gradient accumulation (Effective batch = 16)
LEARNING_RATE   = 3e-4                     # Peak learning rate with cosine decay
AALW_WEIGHT     = 4.0                      # Loss weight on deliberation & answer tokens
SAVE_INTERVAL   = 250                      # Steps per checkpoint save & cloud sync
LOAD_IN_4BIT    = True                     # 4-bit quantization (fits Gemma 2 in ~2.5 GB VRAM)

run_distillation(
    teacher_model_id=TEACHER_MODEL,
    student_preset="v5_small",
    mode=DISTILL_MODE,
    total_steps=TOTAL_STEPS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    aalw_weight=AALW_WEIGHT,
    save_interval=SAVE_INTERVAL,
    cloud_sync=True,
    load_in_4bit=LOAD_IN_4BIT,
)

## ⚡ Step 3: Interactive Deliberation & Reasoning Probe
Test prompt completions from the distilled student model (`v5_small`) to inspect generated `<think> ... </think>` reasoning and answers.

In [ ]:
# ==============================================================================
# 🔍 Step 3: Interactive Prompt Testing on Distilled V5 Small
# ==============================================================================
import torch, os
from transformers import AutoTokenizer
from src.spheroid_v5 import CerebroSpheroidV5, ModelConfigV5, PRESETS_V5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tok_dir = os.path.join(REPO_DIR, "data", "tokenizer")
tokenizer = AutoTokenizer.from_pretrained(tok_dir if os.path.exists(tok_dir) else "meta-llama/Llama-2-7b-hf")

# Load Student Checkpoint
ckpt_candidates = [
    "/content/drive/MyDrive/cerebro_checkpoints/distill_gemma_v5_small/latest.pt",
    "/content/checkpoints/distill_gemma_v5_small/latest.pt",
    os.path.join(REPO_DIR, "checkpoints/distill_gemma_v5_small/latest.pt"),
]
ckpt_path = next((p for p in ckpt_candidates if os.path.exists(p)), None)

cfg = ModelConfigV5(**PRESETS_V5["v5_small"])
model = CerebroSpheroidV5(cfg).to(device)
if ckpt_path:
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt.get("model_state", ckpt), strict=False)
    print(f"✅ Loaded checkpoint from {ckpt_path} (Step {ckpt.get('step', '?')})")
model.eval()

def generate_reasoning(prompt_text, max_tokens=128):
    formatted = f"<stim> {prompt_text.strip()} </stim>"
    input_ids = tokenizer.encode(formatted, add_special_tokens=False)
    p_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)
    out_tokens = []
    with torch.no_grad():
        for _ in range(max_tokens):
            tgt_in = torch.tensor([[1] + out_tokens], dtype=torch.long, device=device) if out_tokens else torch.tensor([[1]], dtype=torch.long, device=device)
            _, logits = model(prompts=p_tensor, targets=tgt_in, deliberation_mode="teacher_forcing")
            next_id = int(torch.argmax(logits[0, -1, :]).item())
            if next_id in (0, 2):
                break
            out_tokens.append(next_id)
    return tokenizer.decode(out_tokens, skip_special_tokens=True)

test_prompt = "If a cell divides every 20 minutes, how many cells will be produced from 1 cell after 2 hours?"
print(f"Prompt:   {test_prompt}")
print(f"Response: {generate_reasoning(test_prompt)}")